# 2차 — Greedy Ensemble Selection (조금씩 쌓아가는 블렌딩)

## 배경
지금까지 "집단 불균형 완화" 단일 기법들(가중치/리샘플링/인터랙션/target encoding/IVF·DI 분리)은
전부 손해였음. 블렌딩도 "전부 한꺼번에 넣고 가중치 최적화"(12개 모델, nested CV 0.74065)는
2개 모델 단순 가중평균(0.74071)보다 못했음 — 후보가 너무 많고 중복돼서 고차원 최적화가 불안정.

이번엔 다른 방식: **Greedy Ensemble Selection** (Caruana 2004 방식) — 후보를 한꺼번에 다 넣지
않고, **한 번에 하나씩, 도움이 될 때만** 앙상블에 추가. "조금씩 완화"를 기법이 아니라 **앙상블
구성 방식 자체**로 구현.

## 알고리즘
1. 전체 후보 중 단독 OOF AUC가 가장 높은 모델 1개로 시작
2. 남은 후보 중 하나씩 "지금 앙상블에 추가했을 때 nested CV AUC가 얼마나 좋아지는지" 테스트
3. 가장 많이 개선시키는 후보를 채택 (개선폭이 노이즈 수준 이하면 거기서 멈춤)
4. 후보가 바닥나거나 더 이상 개선이 없을 때까지 2~3 반복

## 누수 방지
가중치는 매 단계마다 **nested CV**로 (학습 fold에서 가중치 찾고 held-out fold로 평가) — 이전
블렌딩 노트북과 동일한 원칙.

---
**저장 위치:** `experiment_history/2차/2차_greedy_ensemble_selection.ipynb`


In [1]:
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
SUBMIT_DIR = Path("../submit file")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"

N_SPLITS = 5
SEED = 42
IMPROVE_THRESHOLD = 0.00015  # 노이즈(0.00011)보다 살짝 크게 — 이 이상 개선 안 되면 멈춤


## 1. 타깃 로드 + CV 폴드

In [3]:
y = pd.read_csv(TRAIN_PATH)[TARGET_COL].values.astype(int)
test_ids = pd.read_csv(TEST_PATH)["ID"]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(y)), y))
print(f"y 로드 완료: {len(y)}개, 양성비율={y.mean():.4f}")


y 로드 완료: 256351개, 양성비율=0.2583


## 2. blend_cache 자동 스캔 (두 명명규칙 + 팀원 v5 + 안전장치)

In [4]:
MANUAL_TEST_MAP = {
    "oof_10seed_bagged.npy": "test_lgbm_bagged.npy",
}
V5_DIR = SHARED_DIR / "팀원파일" / "김영혜"
V5_OOF_PATH = V5_DIR / "v5_ensemble_oof.npy"
V5_SUBMISSION_PATH = V5_DIR / "submission_v5_imputed.csv"

all_npy = sorted(glob.glob(str(BLEND_DIR / "*.npy")))
oof_candidates = [f for f in all_npy if "oof" in Path(f).stem.lower()]

models = {}
unmatched = []
for f in oof_candidates:
    fname = Path(f).name
    stem = Path(f).stem
    oof_arr = np.load(f)
    if len(oof_arr) != len(y):
        print(f"⚠ 스킵: {fname} (길이 불일치)")
        continue

    if stem.startswith("oof_"):
        name = stem[len("oof_"):]
        default_test = BLEND_DIR / f"test_{name}.npy"
    elif stem.endswith("_oof"):
        name = stem[: -len("_oof")]
        default_test = BLEND_DIR / f"{name}_test.npy"
    else:
        name = stem
        default_test = None

    test_f = BLEND_DIR / MANUAL_TEST_MAP[fname] if fname in MANUAL_TEST_MAP else default_test

    entry = {"oof": oof_arr}
    if test_f is not None and test_f.exists():
        entry["test"] = np.load(test_f)
    else:
        unmatched.append(fname)
    models[name] = entry

if V5_OOF_PATH.exists():
    v5_oof = np.load(V5_OOF_PATH)
    if len(v5_oof) == len(y):
        entry = {"oof": v5_oof}
        if V5_SUBMISSION_PATH.exists():
            entry["test"] = pd.read_csv(V5_SUBMISSION_PATH)["probability"].values
        models["v5_ensemble"] = entry

# 안전장치: 'y' 자체이거나 AUC가 비정상적으로 높은 항목 제외 (타깃 누출 의심)
removed = []
for n in list(models.keys()):
    if n == "y":
        removed.append(n); del models[n]; continue
    auc_check = roc_auc_score(y, models[n]["oof"])
    if auc_check >= 0.995:
        removed.append(n); del models[n]

if removed:
    print(f"⚠ 자동 제외(타깃/누출 의심): {removed}\n")

model_names = list(models.keys())
print(f"발견된 후보: {model_names}")
if unmatched:
    print(f"⚠ test 짝 없음(OOF 비교에만 포함): {unmatched}")
print()
print(f"{'후보':<28} | {'OOF AUC':>8} | {'test':>6}")
print("-" * 48)
for n in model_names:
    auc = roc_auc_score(y, models[n]["oof"])
    has_test = "있음" if "test" in models[n] else "없음"
    print(f"{n:<28} | {auc:>8.5f} | {has_test:>6}")


⚠ 자동 제외(타깃/누출 의심): ['y']

발견된 후보: ['lgbm_lambdarank', '10seed_bagged', 'age_split', 'catboost', 'catboost_bagged', 'ebm', 'feature_subspace', 'lgbm', 'lgbm_keepnan', 'lgbm_robust_search', 'lgbm_tuned', 'mlp', 'tt', 'xgboost', 'xgboost_bagged', 'xgb_rankpairwise', 'v5_ensemble']
⚠ test 짝 없음(OOF 비교에만 포함): ['oof_age_split.npy', 'oof_catboost.npy', 'oof_ebm.npy', 'oof_lgbm.npy', 'oof_lgbm_robust_search.npy', 'oof_lgbm_tuned.npy', 'oof_tt.npy', 'oof_xgboost.npy', 'oof_y.npy']

후보                           |  OOF AUC |   test
------------------------------------------------
lgbm_lambdarank              |  0.73748 |     있음
10seed_bagged                |  0.74036 |     있음
age_split                    |  0.73922 |     없음
catboost                     |  0.73971 |     없음
catboost_bagged              |  0.74005 |     있음
ebm                          |  0.73766 |     없음
feature_subspace             |  0.73973 |     있음
lgbm                         |  0.73925 |     없음
lgbm_keepnan                 |  0

## 3. Nested CV 가중치 탐색 (가벼운 버전 — greedy 반복에 맞게 속도 우선)

In [5]:
def softmax_weights(theta):
    e = np.exp(theta - np.max(theta))
    return e / e.sum()


def optimize_weights(mat, y_sub, n_models, n_restarts=2, seed=0):
    rng = np.random.RandomState(seed)
    best_w, best_auc = None, -1.0
    inits = [np.zeros(n_models)] + [rng.normal(0, 1, n_models) for _ in range(n_restarts - 1)]
    for theta0 in inits:
        res = minimize(lambda t: -roc_auc_score(y_sub, mat @ softmax_weights(t)), theta0,
                        method="Nelder-Mead", options={"maxiter": 1500})
        w = softmax_weights(res.x)
        auc = roc_auc_score(y_sub, mat @ w)
        if auc > best_auc:
            best_auc, best_w = auc, w
    return best_w


def nested_cv_blend_auc(oof_list):
    """oof_list: 블렌딩할 OOF 배열들의 리스트. nested CV로 가중치 찾아 평가한 OOF AUC 반환."""
    mat = np.column_stack(oof_list)
    n_models = mat.shape[1]
    if n_models == 1:
        return roc_auc_score(y, mat[:, 0])
    blended = np.zeros(len(y))
    for tr_idx, val_idx in folds:
        w = optimize_weights(mat[tr_idx], y[tr_idx], n_models)
        blended[val_idx] = mat[val_idx] @ w
    return roc_auc_score(y, blended)


## 4. Greedy Ensemble Selection (한 번에 하나씩 추가)

In [6]:
# test 예측 있는 후보로만 제한 (실제 제출 가능한 조합만 탐색)
model_names = [n for n in model_names if "test" in models[n]]
print(f"test 예측 있는 후보만: {model_names}")


test 예측 있는 후보만: ['lgbm_lambdarank', '10seed_bagged', 'catboost_bagged', 'feature_subspace', 'lgbm_keepnan', 'mlp', 'xgboost_bagged', 'xgb_rankpairwise', 'v5_ensemble']


In [7]:
individual_aucs = {n: roc_auc_score(y, models[n]["oof"]) for n in model_names}
selected = [max(individual_aucs, key=individual_aucs.get)]
remaining = [n for n in model_names if n != selected[0]]
current_auc = individual_aucs[selected[0]]

print(f"시작: {selected[0]} (단독 AUC={current_auc:.5f})\n")

history = [(list(selected), current_auc)]
while remaining:
    best_candidate, best_new_auc = None, current_auc
    for cand in remaining:
        trial_oofs = [models[n]["oof"] for n in selected] + [models[cand]["oof"]]
        trial_auc = nested_cv_blend_auc(trial_oofs)
        if trial_auc > best_new_auc:
            best_new_auc, best_candidate = trial_auc, cand

    improvement = best_new_auc - current_auc
    if best_candidate is None or improvement < IMPROVE_THRESHOLD:
        print(f"중단: 남은 후보 중 {IMPROVE_THRESHOLD} 이상 개선시키는 게 없음 "
              f"(최선이었던 개선폭: {improvement:+.5f})")
        break

    selected.append(best_candidate)
    remaining.remove(best_candidate)
    current_auc = best_new_auc
    history.append((list(selected), current_auc))
    print(f"+ {best_candidate:<28} → nested CV AUC = {current_auc:.5f}  (개선 {improvement:+.5f})")

print(f"\n최종 선택된 앙상블: {selected}")
print(f"최종 nested CV 블렌딩 OOF AUC: {current_auc:.5f}")
print()
print("비교 기준:")
print("  멀티시드 배깅 baseline: 0.74037")
print("  팀 블렌딩(2개 모델, 가중평균): 0.74071")


시작: v5_ensemble (단독 AUC=0.74043)

+ 10seed_bagged                → nested CV AUC = 0.74061  (개선 +0.00018)
중단: 남은 후보 중 0.00015 이상 개선시키는 게 없음 (최선이었던 개선폭: +0.00001)

최종 선택된 앙상블: ['v5_ensemble', '10seed_bagged']
최종 nested CV 블렌딩 OOF AUC: 0.74061

비교 기준:
  멀티시드 배깅 baseline: 0.74037
  팀 블렌딩(2개 모델, 가중평균): 0.74071


## 5. 최종 선택된 조합으로 test 예측 생성

In [8]:
models_with_test = [n for n in selected if "test" in models[n]]
missing = set(selected) - set(models_with_test)
if missing:
    print(f"⚠ test 예측 없어서 최종 제출에서 제외됨: {missing}")

oof_sub = np.column_stack([models[n]["oof"] for n in models_with_test])
test_sub = np.column_stack([models[n]["test"] for n in models_with_test])
n_sub = len(models_with_test)

w_final = optimize_weights(oof_sub, y, n_sub, n_restarts=5)
test_pred_final = test_sub @ w_final
final_oof_auc = roc_auc_score(y, oof_sub @ w_final)

print(f"최종 가중치: {dict(zip(models_with_test, np.round(w_final, 4)))}")
print(f"최종(전체 train으로 fit) OOF AUC: {final_oof_auc:.5f}")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "blend_greedy_test.npy", test_pred_final)

SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": test_pred_final})
out_path = SUBMIT_DIR / f"2차_blend_greedy_local{final_oof_auc:.5f}.csv"
submission.to_csv(out_path, index=False)
print(f"비제출용 결과 저장: {out_path}")


최종 가중치: {'v5_ensemble': np.float64(0.5192), '10seed_bagged': np.float64(0.4808)}
최종(전체 train으로 fit) OOF AUC: 0.74066
비제출용 결과 저장: ../submit file/2차_blend_greedy_local0.74066.csv


## 6. 해석

- 최종 nested CV AUC가 0.74071을 의미있게(노이즈 0.00011의 2~3배 이상) 넘으면 — 진짜 개선,
  이 조합을 제출 후보로 검토
- 0.74071 근처에서 멈췄다면 — greedy 방식으로도 그 이상은 안 나온다는 추가 확인, 0.74071이
  이 데이터셋에서 만들 수 있는 거의 최선이라는 결론을 더 굳힐 수 있음
- `history` 변수에 단계별 선택 과정이 다 남아있으니, 어떤 후보가 어느 시점에 추가됐는지(또는
  전혀 선택 안 됐는지) 살펴보면 "이 모델/기법은 앙상블에 기여하지 못한다"를 후보별로 명확히
  결론 내릴 수 있음
